# Stage 26A: Zero-Dose Identity Audit

We discovered in Stage 26 that at $\rho=0$, the intervened gradient update does not perfectly reproduce the natural update ($E_{\text{update}}$ drifts up to $10^{-2}$). 

We hypothesize this is a numerical distortion: as training converges, the gradient difference $\delta_m$ approaches zero. The $F_{10}$ preservation solver uses `+ 1e-12` regularization in its denominators. When $\|\delta_m\|^2 \approx 10^{-12}$, this regularizer heavily distorts the scaling coefficients $c_T, c_N$, injecting artificial perturbation into the "zero-dose" condition.

This notebook runs a focused $\rho=0$ diagnostic on a few seeds and K values, recording:
1. $E_\delta = \frac{\|\delta_{\text{corrected}} - \delta_m\|}{\|\delta_m\|}$
2. $E_g = \frac{\|g_{\text{int}} - g_{\text{nat}}\|}{\|g_{\text{nat}}\|}$
3. $E_{\text{update}} = \frac{\|d_{\text{int}} - d_{\text{nat}}\|}{\|d_{\text{nat}}\|}$

If $E_\delta \approx 10^{-2}$, it confirms the solver is self-corrupting at small scales. We will then correct the scaling logic.

In [ ]:
import json
import math
import sys
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn

sys.path.append('.')
from official_soap import SOAP


In [ ]:
class SOAPStage26A(SOAP):
    def __init__(self, params, rho=0.0, base_seed=0, lr=1e-3, betas=(0.95, 0.95), shampoo_beta=-1, 
                 eps=1e-8, weight_decay=0.0, precondition_frequency=10, 
                 max_precond_dim=10000, merge_dims=False, precondition_1d=False,
                 normalize_grads=False, data_format='channels_first', correct_bias=True,
                 mu=0.5):
        super().__init__(params, lr=lr, betas=betas, shampoo_beta=shampoo_beta, 
                         eps=eps, weight_decay=weight_decay, 
                         precondition_frequency=precondition_frequency,
                         max_precond_dim=max_precond_dim, merge_dims=merge_dims, 
                         precondition_1d=precondition_1d, normalize_grads=normalize_grads, 
                         data_format=data_format, correct_bias=correct_bias)
        self.rho = rho
        self.base_seed = base_seed
        self.mu = mu
        self.diagnostics = {}

    def functional_soap_update(self, raw_gradient, state, group, p):
        b1, b2 = group['betas']
        eps = group['eps']
        
        proj_grad = self.project(raw_gradient.clone(), state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
        
        m_old = state.get("exp_avg", torch.zeros_like(proj_grad))
        v_old = state.get("exp_avg_sq", torch.zeros_like(proj_grad))
        
        m_new = m_old * b1 + proj_grad * (1.0 - b1)
        v_new = v_old * b2 + proj_grad.square() * (1.0 - b2)
        
        denom = v_new.sqrt() + eps
        
        step_val = state.get('step', 0) + 1
        step_size = group['lr']
        if getattr(self, "correct_bias", True):
            bias_correction1 = 1.0 - b1 ** step_val
            bias_correction2 = 1.0 - b2 ** step_val
            step_size = step_size * (math.sqrt(bias_correction2)) / bias_correction1
            
        norm_grad = self.project_back(m_new / denom, state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
        
        if group.get("normalize_grads", False):
            norm_grad = norm_grad / (1e-30 + torch.mean(norm_grad**2)**0.5)
            
        update = -step_size * norm_grad
        
        if group.get("weight_decay", 0.0) > 0.0:
            update = update - group['lr'] * group['weight_decay'] * p.data
            
        return update

    @torch.no_grad()
    def step(self, closure=None):
        loss = None if closure is None else closure()
        self.diagnostics = {"count": 0, "layers": {}}
        
        current_step = 0
        if len(self.param_groups) > 0 and len(self.param_groups[0]['params']) > 0:
            p_sample = self.param_groups[0]['params'][0]
            if 'step' in self.state[p_sample]:
                current_step = self.state[p_sample]['step'] + 1
        else:
            current_step = 1
        
        idx = 0
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    idx += 1
                    continue
                
                layer_idx = idx
                state = self.state[p]
                raw_g = p.grad.clone()
                
                if 'moq_initialized' not in state:
                    state['moq_initialized'] = True
                    state['prev_raw_g'] = raw_g.clone()
                    g_final = raw_g.clone()
                else:
                    g_nat = raw_g + self.mu * (raw_g - state['prev_raw_g'])
                    
                    proj_raw = self.project(raw_g.clone(), state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
                    b1, b2 = group["betas"]
                    v_old = state.get("exp_avg_sq")
                    
                    if v_old is not None and p.dim() > 1 and 'Q' in state and state['Q'] is not None:
                        v_raw = v_old * b2 + proj_raw.square() * (1.0 - b2)
                        denom_raw = v_raw.sqrt() + group["eps"]
                        D_inv = 1.0 / denom_raw
                        
                        delta_g_orig = self.mu * (raw_g - state['prev_raw_g'])
                        proj_delta = self.project(delta_g_orig.clone(), state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
                        
                        delta_m = proj_delta * (1.0 - b1)
                        delta_m_norm = float(torch.linalg.vector_norm(delta_m))
                        
                        if delta_m_norm > 1e-12:
                            denom_flat = denom_raw.flatten()
                            k10 = max(1, int(0.10 * denom_flat.numel()))
                            _, indices10 = torch.topk(denom_flat, k10, largest=False)
                            
                            tail_mask = torch.zeros_like(denom_flat, dtype=torch.bool)
                            tail_mask[indices10] = True
                            
                            # -----------------------------------------------------
                            # EXACT STAGE 26 CODE FOR RECONSTRUCTION
                            # -----------------------------------------------------
                            z_delta = D_inv * delta_m
                            z_delta_sq = z_delta.flatten()**2
                            sum_z_delta_sq = float(torch.sum(z_delta_sq)) + 1e-12
                            F10_nat = float(torch.sum(z_delta_sq[indices10])) / sum_z_delta_sq
                            
                            delta_m_flat = delta_m.flatten()
                            d_T_vals = delta_m_flat[tail_mask]
                            d_N_vals = delta_m_flat[~tail_mask]
                            
                            if len(d_T_vals) > 0 and len(d_N_vals) > 0:
                                
                                seed_val = self.base_seed + current_step * 1000 + layer_idx
                                gen = torch.Generator(device='cpu')
                                gen.manual_seed(seed_val)
                                
                                perm_T = torch.randperm(len(d_T_vals), generator=gen, device='cpu').to(d_T_vals.device)
                                perm_N = torch.randperm(len(d_N_vals), generator=gen, device='cpu').to(d_N_vals.device)
                                
                                d_T_perm = d_T_vals[perm_T]
                                d_N_perm = d_N_vals[perm_N]
                                
                                d_T_mix = (1.0 - self.rho) * d_T_vals + self.rho * d_T_perm
                                d_N_mix = (1.0 - self.rho) * d_N_vals + self.rho * d_N_perm
                                
                                d_perm_flat = torch.zeros_like(delta_m_flat)
                                d_perm_flat[tail_mask] = d_T_mix
                                d_perm_flat[~tail_mask] = d_N_mix
                                d_perm = d_perm_flat.view(delta_m.shape)
                                
                                tail_mask_shaped = tail_mask.view(denom_raw.shape)
                                d_T_full = torch.where(tail_mask_shaped, d_perm, torch.zeros_like(d_perm))
                                d_N_full = torch.where(~tail_mask_shaped, d_perm, torch.zeros_like(d_perm))
                                
                                norm_d_T_sq = float(torch.sum(d_T_full**2))
                                norm_d_N_sq = float(torch.sum(d_N_full**2))
                                
                                E_T = float(torch.sum((D_inv * d_T_full)**2))
                                E_N = float(torch.sum((D_inv * d_N_full)**2))
                                
                                if F10_nat == 0.0:
                                    c_T = 0.0
                                    c_N = 1.0 / math.sqrt(norm_d_N_sq + 1e-12)
                                elif F10_nat == 1.0:
                                    c_T = 1.0 / math.sqrt(norm_d_T_sq + 1e-12)
                                    c_N = 0.0
                                else:
                                    ratio = (F10_nat * E_N) / ((1.0 - F10_nat) * E_T + 1e-12)
                                    c_N_sq = 1.0 / (ratio * norm_d_T_sq + norm_d_N_sq + 1e-12)
                                    c_T_sq = c_N_sq * ratio
                                    c_N = math.sqrt(c_N_sq)
                                    c_T = math.sqrt(c_T_sq)
                                    
                                d_corrected = (c_T * d_T_full + c_N * d_N_full) * delta_m_norm
                                proj_delta_corrected = d_corrected / (1.0 - b1)
                                delta_g_corrected = self.project_back(proj_delta_corrected, state, merge_dims=group["merge_dims"], max_precond_dim=group['max_precond_dim'])
                                g_int = raw_g + delta_g_corrected
                                g_final = g_int
                                
                                # -----------------------------------------------------
                                # AUDIT LOGIC
                                # -----------------------------------------------------
                                E_delta = float(torch.linalg.vector_norm(proj_delta_corrected - proj_delta)) / (float(torch.linalg.vector_norm(proj_delta)) + 1e-12)
                                E_g = float(torch.linalg.vector_norm(g_int - g_nat)) / (float(torch.linalg.vector_norm(g_nat)) + 1e-12)
                                
                                d_nat = self.functional_soap_update(g_nat, state, group, p)
                                d_int = self.functional_soap_update(g_int, state, group, p)
                                
                                E_up = float(torch.linalg.vector_norm(d_int - d_nat)) / (float(torch.linalg.vector_norm(d_nat)) + 1e-12)
                                
                                self.diagnostics["layers"][layer_idx] = {
                                    "E_delta": E_delta,
                                    "E_g": E_g,
                                    "E_update": E_up,
                                    "delta_m_norm": delta_m_norm
                                }
                                
                                self.diagnostics["count"] += 1
                            else:
                                g_final = g_nat.clone()
                        else:
                            g_final = g_nat.clone()
                    else:
                        g_final = g_nat.clone()
                            
                    state['prev_raw_g'].copy_(raw_g)
                
                p.grad.copy_(g_final)
                idx += 1
                
        super().step()
        return loss


In [ ]:
class PINN(nn.Module):
    def __init__(self, width: int = 128, hidden_layers: int = 4) -> None:
        super().__init__()
        layers: List[nn.Module] = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        return self.net(xy)

def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
def sample_collocation(n: int, device: torch.device) -> torch.Tensor:
    return torch.rand(n, 2, device=device, requires_grad=True)

def sample_boundary(n_each: int, device: torch.device) -> torch.Tensor:
    t = torch.rand(4 * n_each, device=device)
    x = torch.empty(4 * n_each, 2, device=device)
    x[:n_each] = torch.stack([torch.zeros(n_each, device=device), t[:n_each]], dim=1)
    x[n_each:2*n_each] = torch.stack([torch.ones(n_each, device=device), t[n_each:2*n_each]], dim=1)
    x[2*n_each:3*n_each] = torch.stack([t[2*n_each:3*n_each], torch.zeros(n_each, device=device)], dim=1)
    x[3*n_each:] = torch.stack([t[3*n_each:], torch.ones(n_each, device=device)], dim=1)
    return x

def make_batch_schedule(steps: int, K: int, n_pde: int, n_boundary_each: int, device: torch.device) -> List[Dict[str, torch.Tensor]]:
    schedule: List[Dict[str, torch.Tensor]] = []
    pde: Optional[torch.Tensor] = None
    bc: Optional[torch.Tensor] = None
    for step in range(1, steps + 1):
        if step == 1 or (step - 1) % K == 0:
            pde = sample_collocation(n_pde, device)
            bc = sample_boundary(n_boundary_each, device)
        schedule.append({
            "pde": pde.detach().clone(),
            "bc": bc.detach().clone(),
        })
    return schedule

def poisson_residual(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    u = model(xy)
    grad_u = torch.autograd.grad(u, xy, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]
    u_x, u_y = grad_u[:, 0:1], grad_u[:, 1:2]
    u_xx = torch.autograd.grad(u_x, xy, grad_outputs=torch.ones_like(u_x), create_graph=True, retain_graph=True)[0][:, 0:1]
    u_yy = torch.autograd.grad(u_y, xy, grad_outputs=torch.ones_like(u_y), create_graph=True, retain_graph=True)[0][:, 1:2]
    forcing = 2.0 * math.pi**2 * torch.sin(math.pi * xy[:, 0:1]) * torch.sin(math.pi * xy[:, 1:2])
    return -(u_xx + u_yy) - forcing


In [ ]:
def run_stage26A_audit(K: int=10, seed: int=0):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = PINN().to(device)
    
    schedule = make_batch_schedule(2000, K, 1000, 100, device)

    opt = SOAPStage26A(
        model.parameters(), 
        rho=0.0,
        base_seed=seed,
        lr=1e-3, 
        betas=(0.95, 0.95), 
        precondition_frequency=10, 
        mu=0.5
    )

    audit_steps = {10, 100, 500, 1000, 1500, 2000}
    audit_results = {}
    
    model.train()
    for step in range(1, 2001):
        current_pde = schedule[step - 1]["pde"].to(device).detach().clone().requires_grad_(True)
        current_bc = schedule[step - 1]["bc"].to(device)
        opt.zero_grad(set_to_none=True)

        r = poisson_residual(model, current_pde)
        loss = torch.mean(r.square()) + torch.mean(model(current_bc).square())
        loss.backward()
        opt.step()
        
        if step in audit_steps:
            audit_results[step] = opt.diagnostics["layers"]
            
    return audit_results


### Execute Audit

In [ ]:
Ks = [10, 20]
seeds = [0, 1]

all_audit_results = {}

for K in Ks:
    all_audit_results[K] = {}
    for seed in seeds:
        print(f"\n{'='*40}\nRunning K={K} Seed={seed} Audit...\n{'='*40}")
        results = run_stage26A_audit(K=K, seed=seed)
        all_audit_results[K][seed] = results
        
with open(f"stage26A_audit.json", "w") as f:
    json.dump(all_audit_results, f, indent=2)
